In [1]:
import math
import pandas as pd
import numpy as np
import datetime

In [2]:
df = pd.read_excel(r"D:\Reading Material\Statistics\Descriptive Stats\Contact Center Operational Analytics\01 Call-Center-Dataset.xlsx", sheet_name = "Sheet1")
df.head()

,Call Id,Agent,Date,Time,Topic,Answered (Y/N),Resolved,Speed of answer in seconds,AvgTalkDuration,Satisfaction rating
0,ID0001,Diane,2021-01-01,09:12:58,Contract related,Y,Y,109.0,00:02:23,3.0
1,ID0002,Becky,2021-01-01,09:12:58,Technical Support,Y,N,70.0,00:04:02,3.0
2,ID0003,Stewart,2021-01-01,09:47:31,Contract related,Y,Y,10.0,00:02:11,3.0
3,ID0004,Greg,2021-01-01,09:47:31,Contract related,Y,Y,53.0,00:00:37,2.0
4,ID0005,Becky,2021-01-01,10:00:29,Payment related,Y,Y,95.0,00:01:00,3.0


In [3]:
# Business Question 1
# How is the overall performance of the contact center?

In [4]:
# Total calls
total_calls = df['Call Id'].count()
print("Total calls:", total_calls)

Total calls: 5000


In [5]:
# Answered Calls
answered_calls = (df['Answered (Y/N)'] == "Y").sum()
print("Total Answered calls:", answered_calls)

# Unanswered Calls
Unanswered_calls = (df['Answered (Y/N)'] == "N").sum()
print("Total Unanswered calls:", Unanswered_calls)

Total Answered calls: 4054
Total Unanswered calls: 946


In [6]:
df.columns

Index(['Call Id', 'Agent', 'Date', 'Time', 'Topic', 'Answered (Y/N)',
       'Resolved', 'Speed of answer in seconds', 'AvgTalkDuration',
       'Satisfaction rating'],
      dtype='str')

In [7]:
# Answer Rate
answer_rate = (answered_calls / total_calls) * 100
print(f"Answered Rate: {answer_rate:.2f}%")

Answered Rate: 81.08%


In [8]:
# Total Resolved Calls
resolved_calls = (df['Resolved'] == "Y").sum()
print("Total Resolved calls:", resolved_calls)

Total Resolved calls: 3646


In [9]:
# Resolution Rate
resolution_rate = (resolved_calls / total_calls) * 100
print(f"Resolution Rate: {resolution_rate:.2f}%")

Resolution Rate: 72.92%


In [10]:
# Average Speed Of Answer
avg_answer_speed = (df['Speed of answer in seconds']).mean()
print(f"Avg Speed of Answer: {avg_answer_speed:.2f} sec")

Avg Speed of Answer: 67.52 sec


In [11]:
# Average Talk Duration
from datetime import datetime, timedelta

# ✅ Step 1: Ensure values are strings
df['AvgTalkDuration'] = df['AvgTalkDuration'].astype(str)

# Convert talk duration to timedelta
df['AvgTalkDuration'] = pd.to_timedelta(df['AvgTalkDuration'])

# Average Talk Duration for Answered Calls only
avg_talk_duration = (df.loc[df['Answered (Y/N)'] == 'Y', 'AvgTalkDuration'].mean())

# Convert average duration to minutes and seconds
total_seconds = int(avg_talk_duration.total_seconds())

minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

print(f"Average Talk Duration: {minutes} min {seconds} sec")

Average Talk Duration: 3 min 44 sec


In [12]:
# Average Rating Score
CSAT_Score = (df['Satisfaction rating']).mean()
print(f"CSAT Score: {CSAT_Score:.2f}")

CSAT Score: 3.40


In [13]:
# Business Question 2
# How consistent is contact-center performance?
# a. Spped of Answer

In [14]:
answer_speed_stats = df.loc[df['Answered (Y/N)'] == "Y", 'Speed of answer in seconds'].describe()
print(answer_speed_stats)

count    4054.000000
mean       67.520720
std        33.592872
min        10.000000
25%        39.000000
50%        68.000000
75%        97.000000
max       125.000000
Name: Speed of answer in seconds, dtype: float64


In [15]:
# The mean (67.52 sec) and median (68 sec) are almost identical, so the distribution is reasonably 
# centered around ~68 seconds.

# However, the standard deviation of 33.59 seconds is relatively large compared with the mean. 
# This tells us that answering speed varies significantly from call to call.

# The interquartile range is: 97 − 39 = 58 seconds
# So the middle 50% of answered calls took 39–97 seconds to answer.

# Need to find : Why are some calls answered much faster or slower than others?(Driver Analysis / Root Cause Analysis)

In [16]:
# Calculate variance

speed_variance = df.loc[df['Answered (Y/N)'] == 'Y', 'Speed of answer in seconds'].var()
print(f"Speed of Answer Variance: {speed_variance:.2f}")

Speed of Answer Variance: 1128.48


In [17]:
# Speed of Answer averaged 67.5 seconds, with a standard deviation of 33.6 seconds, 
# indicating substantial variation in how quickly calls were answered.

In [18]:
# Percentiles

speed_percentiles = df.loc[df['Answered (Y/N)'] == 'Y', 'Speed of answer in seconds'].quantile([0.90, 0.95])
print(speed_percentiles)

0.90    114.0
0.95    120.0
Name: Speed of answer in seconds, dtype: float64


In [19]:
# Answering performance varies considerably across calls. While the typical call is answered in approximately 68 seconds, 
# the middle 50% range from 39 to 97 seconds, and 5% of answered calls take 120 seconds or longer.

In [20]:
# 2.b. calculate talk duration

# ✅ Step 1: Ensure values are strings
df['AvgTalkDuration'] = df['AvgTalkDuration'].astype(str)

# Convert talk duration to timedelta
df['AvgTalkDuration'] = pd.to_timedelta(df['AvgTalkDuration'])

# Talk Duration for Answered Calls only
talk_duration = (df.loc[df['Answered (Y/N)'] == 'Y', 'AvgTalkDuration'].dt.total_seconds())
talk_duration.describe()

count    4054.000000
mean      224.922792
std       111.381555
min        30.000000
25%       130.000000
50%       226.000000
75%       319.000000
max       420.000000
Name: AvgTalkDuration, dtype: float64

In [21]:
# Average talk duration is approximately 3 minutes 45 seconds, with substantial variation across calls. 
# The middle 50% of calls have talk durations between 130 and 319 seconds, indicating that call-handling time 
# differs considerably across interactions.

In [22]:
# Business conclusion:
# Contact-center performance shows substantial call-level variation in both answering speed and handling time. 
# Speed of Answer averages 67.5 seconds, while Talk Duration averages 224.9 seconds, with wide distributions around 
# both measures. This suggests that operational performance should be examined further by agent, topic, and other 
# call characteristics to identify the sources of variation.

In [23]:
# Business Question 3: (Trend Analysis)
# Is contact-center performance improving, deteriorating, or remaining stable over time?

In [24]:
df['Date'] = pd.to_datetime(df['Date'])

monthly_trend = df.groupby(
    df['Date'].dt.to_period('M')
).agg(
    Call_Volume=('Call Id', 'count'),
    Answered_Calls=('Answered (Y/N)', lambda x: (x == 'Y').sum()),
    Resolved_Calls=('Resolved', lambda x: (x == 'Y').sum()),
    Avg_Speed_of_Answer=('Speed of answer in seconds', 'mean'),
    Avg_Satisfaction=('Satisfaction rating', 'mean')
).reset_index()

monthly_trend['Answer_Rate'] = (
    monthly_trend['Answered_Calls']
    / monthly_trend['Call_Volume'] * 100
)

monthly_trend['Resolution_Rate'] = (
    monthly_trend['Resolved_Calls']
    / monthly_trend['Call_Volume'] * 100
)

monthly_trend

,Date,Call_Volume,Answered_Calls,Resolved_Calls,Avg_Speed_of_Answer,Avg_Satisfaction,Answer_Rate,Resolution_Rate
0,2021-01,1772,1455,1311,67.219931,3.454296,82.110609,73.984199
1,2021-02,1616,1298,1161,67.546225,3.377504,80.321782,71.844059
2,2021-03,1612,1301,1174,67.831668,3.372790,80.707196,72.828784


In [25]:
# Business conclusion:
# Contact-center performance weakened after January. Answer Rate and Resolution Rate both declined in February 
# and only partially recovered in March, while customer satisfaction continued to decline. 
# Average Speed of Answer remained broadly stable, suggesting that factors beyond response time may be contributing 
# to the deterioration in customer experience.

In [26]:
# Question 4 — Which agents are performing differently from the overall operation?
# Comparative Analysis + Descriptive Analysis
# Performance benchmarking against the overall operation

In [27]:
# Business purpose:
# Management wants to know whether performance variation is concentrated among particular agents 
# and which agents may require attention or recognition.

In [28]:
# Create agent-level performance summary

agents = df['Agent'].unique()

agent_results = []

for agent in agents:

    agent_df = df[df['Agent'] == agent]
    answered_df = agent_df[agent_df['Answered (Y/N)'] == 'Y']

    total_calls = len(agent_df)
    answered_calls = len(answered_df)
    resolved_calls = (agent_df['Resolved'] == 'Y').sum()

    answer_rate = answered_calls / total_calls * 100
    resolution_rate = resolved_calls / total_calls * 100

    avg_speed = answered_df['Speed of answer in seconds'].mean()

    talk_duration_seconds = pd.to_timedelta(answered_df['AvgTalkDuration']).dt.total_seconds()

    avg_talk = talk_duration_seconds.mean()

    avg_satisfaction = answered_df['Satisfaction rating'].mean()

    agent_results.append([
        agent,
        total_calls,
        answered_calls,
        resolved_calls,
        answer_rate,
        resolution_rate,
        avg_speed,
        avg_talk,
        avg_satisfaction])

agent_performance = pd.DataFrame(
    agent_results,
    columns=[
        'Agent',
        'Calls',
        'Answered Calls',
        'Resolved Calls',
        'Answer Rate %',
        'Resolution Rate %',
        'Avg Speed of Answer',
        'Avg Talk Duration (sec)',
        'Avg Satisfaction'])

agent_performance = agent_performance.sort_values('Resolution Rate %', ascending=False)

print(agent_performance.round(2))

     Agent  Calls  Answered Calls  Resolved Calls  Answer Rate %  \
7      Dan    633             523             471          82.62   
5      Joe    593             484             436          81.62   
1    Becky    631             517             462          81.93   
3     Greg    624             502             455          80.45   
2  Stewart    582             477             424          81.96   
4      Jim    666             536             485          80.48   
6   Martha    638             514             461          80.56   
0    Diane    633             501             452          79.15   

   Resolution Rate %  Avg Speed of Answer  Avg Talk Duration (sec)  \
7              74.41                67.28                   231.19   
5              73.52                70.99                   224.10   
1              73.22                65.33                   220.01   
3              72.92                68.44                   226.80   
2              72.85                6

In [29]:
# Business Insight:
# 1. Dan is the strongest overall performer
# Dan has the highest resolution rate (74.41%) and a strong 82.62% answer rate. 
# His satisfaction is also above the overall average at 3.45

# 2. Diane needs the most attention on resolution and accessibility
# Lowest Answer Rate: 79.15%
# Lowest Resolution Rate: 71.41%

# Speed of Answer = 66.27 sec
# Talk Duration = 218.95 sec
#Satisfaction = 3.41
# are not poor.

# 3. Martha's satisfaction is 3.47, the highest among the agents.
# But her Speed of Answer is 69.49 sec, which is slower than several other agents.

# 4. Joe's average Speed of Answer is 70.99 sec, the highest among the eight agents.
# His satisfaction is also the lowest at 3.33.

In [30]:
# Management takeaway:
# Agent performance varies across different dimensions, indicating that improvement actions should be 
# targeted rather than applying the same intervention to every agent. Diane's lower resolution/answer rates, 
# Joe's slower response and lower satisfaction, and Martha's comparatively strong satisfaction despite slower 
# response warrant deeper analysis of call topics and performance drivers.

In [31]:
# Question 5 — Which call topics are creating performance challenges?
# Comparative Analysis + Root Cause Analysis

In [34]:
topics = df['Topic'].unique()

topic_results = []

for topic in topics:

    topic_df = df[df['Topic'] == topic]
    answered_df = topic_df[topic_df['Answered (Y/N)'] == 'Y']

    total_calls = len(topic_df)
    answered_calls = len(answered_df)
    resolved_calls = (topic_df['Resolved'] == 'Y').sum()

    answer_rate = answered_calls / total_calls * 100
    resolution_rate = resolved_calls / total_calls * 100

    avg_speed = answered_df['Speed of answer in seconds'].mean()

    talk_duration_seconds = pd.to_timedelta(
        answered_df['AvgTalkDuration']
    ).dt.total_seconds()

    avg_talk = talk_duration_seconds.mean()

    avg_satisfaction = answered_df['Satisfaction rating'].mean()

    topic_results.append([
        topic,
        total_calls,
        answer_rate,
        resolution_rate,
        avg_speed,
        avg_talk,
        avg_satisfaction])

topic_performance = pd.DataFrame(
    topic_results,
    columns=[
        'Topic',
        'Calls',
        'Answer Rate %',
        'Resolution Rate %',
        'Avg Speed of Answer',
        'Avg Talk Duration (sec)',
        'Avg Satisfaction'])

topic_performance = topic_performance.sort_values('Resolution Rate %', ascending=False)

print(topic_performance.round(2))

               Topic  Calls  Answer Rate %  Resolution Rate %  \
3      Admin Support    976          81.45              74.08   
4          Streaming   1022          82.88              73.29   
0   Contract related    976          80.84              72.64   
2    Payment related   1007          81.23              72.39   
1  Technical Support   1019          79.00              72.23   

   Avg Speed of Answer  Avg Talk Duration (sec)  Avg Satisfaction  
3                67.25                   228.05              3.43  
4                66.89                   227.73              3.40  
0                67.16                   228.01              3.38  
2                68.20                   215.86              3.40  
1                68.11                   225.06              3.41  


In [35]:
# Business Question 6: What factors are associated with customer satisfaction?
# Analysis type: Driver Analysis

In [36]:
# Convert AvgTalkDuration into seconds
def convert_to_seconds(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, datetime.time):
        return (
            value.hour * 3600
            + value.minute * 60
            + value.second
            + value.microsecond / 1000000
        )

    return pd.to_timedelta(value).total_seconds()

df["Talk Duration Seconds"] = df["AvgTalkDuration"].apply(convert_to_seconds)

# Only answered calls have satisfaction ratings
answered_df = df[df["Answered (Y/N)"] == "Y"].copy()

# Numeric relationship with satisfaction
correlation = answered_df[
    [
        "Speed of answer in seconds",
        "Talk Duration Seconds",
        "Satisfaction rating"
    ]].corr()

print("Correlation with Satisfaction:")
print(
    correlation["Satisfaction rating"]
    .drop("Satisfaction rating")
    .round(3))

# Satisfaction by resolution
satisfaction_resolution = (
    answered_df
    .groupby("Resolved")["Satisfaction rating"]
    .agg(["count", "mean"])
    .round(2))

print("\nSatisfaction by Resolution:")
print(satisfaction_resolution)

# Satisfaction by topic
satisfaction_topic = (
    answered_df
    .groupby("Topic")["Satisfaction rating"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
    .round(2))

print("\nSatisfaction by Topic:")
print(satisfaction_topic)

# Satisfaction by agent
satisfaction_agent = (
    answered_df
    .groupby("Agent")["Satisfaction rating"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
    .round(2))

print("\nSatisfaction by Agent:")
print(satisfaction_agent)

Correlation with Satisfaction:
Speed of answer in seconds    0.001
Talk Duration Seconds         0.000
Name: Satisfaction rating, dtype: float64

Satisfaction by Resolution:
          count  mean
Resolved             
N           408  3.43
Y          3646  3.40

Satisfaction by Topic:
                   count  mean
Topic                         
Admin Support        795  3.43
Technical Support    805  3.41
Streaming            847  3.40
Payment related      818  3.40
Contract related     789  3.38

Satisfaction by Agent:
         count  mean
Agent               
Martha     514  3.47
Dan        523  3.45
Diane      501  3.41
Greg       502  3.40
Stewart    477  3.40
Jim        536  3.39
Becky      517  3.37
Joe        484  3.33


In [ ]:
# Business Insight
# Customer satisfaction is not materially associated with Speed of Answer or Talk Duration in this dataset. 
# Topic and agent differences are also relatively small, although Joe has the lowest average satisfaction 
# at 3.33 and Martha the highest at 3.47.

In [ ]:
# Business Question 7: What factors are associated with call resolution?
# Analysis type: Driver Analysis + Root Cause Analysis

In [37]:
def convert_to_seconds(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, datetime.time):
        return (
            value.hour * 3600
            + value.minute * 60
            + value.second
            + value.microsecond / 1000000
        )

    return pd.to_timedelta(value).total_seconds()


df["Talk Duration Seconds"] = df["AvgTalkDuration"].apply(convert_to_seconds)

# Analyse answered calls only
answered_df = df[df["Answered (Y/N)"] == "Y"].copy()

# Convert resolution to numeric
answered_df["Resolved Numeric"] = (answered_df["Resolved"] == "Y").astype(int)

# Numeric relationships with resolution
correlation = answered_df[
    [
        "Speed of answer in seconds",
        "Talk Duration Seconds",
        "Satisfaction rating",
        "Resolved Numeric"
    ]].corr()

print("Correlation with Resolution:")
print(
    correlation["Resolved Numeric"]
    .drop("Resolved Numeric")
    .round(3))

# Resolution by topic
resolution_topic = (
    answered_df
    .groupby("Topic")["Resolved Numeric"]
    .mean()
    .mul(100)
    .sort_values()
    .round(2))

print("\nResolution Rate by Topic:")
print(resolution_topic)

# Resolution by agent
resolution_agent = (
    answered_df
    .groupby("Agent")["Resolved Numeric"]
    .mean()
    .mul(100)
    .sort_values()
    .round(2))

print("\nResolution Rate by Agent:")
print(resolution_agent)

Correlation with Resolution:
Speed of answer in seconds    0.007
Talk Duration Seconds         0.007
Satisfaction rating          -0.007
Name: Resolved Numeric, dtype: float64

Resolution Rate by Topic:
Topic
Streaming            88.43
Payment related      89.12
Contract related     89.86
Admin Support        90.94
Technical Support    91.43
Name: Resolved Numeric, dtype: float64

Resolution Rate by Agent:
Agent
Stewart    88.89
Becky      89.36
Martha     89.69
Dan        90.06
Joe        90.08
Diane      90.22
Jim        90.49
Greg       90.64
Name: Resolved Numeric, dtype: float64


In [ ]:
# Business Insight:

# Agent resolution rates range from approximately:
# Stewart: 88.89%
# Greg: 90.64%

# The biggest variation appears to be by call topic, with Streaming having the lowest answered-call 
# resolution rate at 88.43%, compared with 91.43% for Technical Support. 
# Agent-level differences are much smaller.
# Therefore, the stronger area for root-cause investigation is call type/topic rather than simply 
# agent performance.

In [ ]:
# Business Question 8: Why are some calls taking longer to handle?
# Analysis type: Root Cause Analysis + Driver Analysis

In [38]:
def convert_to_seconds(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, datetime.time):
        return (
            value.hour * 3600
            + value.minute * 60
            + value.second
            + value.microsecond / 1000000
        )

    return pd.to_timedelta(value).total_seconds()


df["Talk Duration Seconds"] = df["AvgTalkDuration"].apply(convert_to_seconds)

answered_df = df[df["Answered (Y/N)"] == "Y"].copy()

# Overall talk duration
print("Overall Average Talk Duration:")
print(round(answered_df["Talk Duration Seconds"].mean(), 2), "seconds")

# Talk duration by topic
talk_topic = (
    answered_df
    .groupby("Topic")["Talk Duration Seconds"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
    .round(2))

print("\nTalk Duration by Topic:")
print(talk_topic)

# Talk duration by agent
talk_agent = (
    answered_df
    .groupby("Agent")["Talk Duration Seconds"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
    .round(2))

print("\nTalk Duration by Agent:")
print(talk_agent)

# Numeric relationships
correlation = answered_df[
    [
        "Speed of answer in seconds",
        "Talk Duration Seconds",
        "Satisfaction rating"
    ]].corr()

print("\nCorrelation with Talk Duration:")
print(
    correlation["Talk Duration Seconds"]
    .drop("Talk Duration Seconds")
    .round(3))

Overall Average Talk Duration:
224.92 seconds

Talk Duration by Topic:
                   count    mean
Topic                           
Admin Support        795  228.05
Contract related     789  228.01
Streaming            847  227.73
Technical Support    805  225.06
Payment related      818  215.86

Talk Duration by Agent:
         count    mean
Agent                 
Dan        523  231.19
Jim        536  228.11
Greg       502  226.80
Stewart    477  226.21
Joe        484  224.10
Martha     514  223.73
Becky      517  220.01
Diane      501  218.95

Correlation with Talk Duration:
Speed of answer in seconds   -0.003
Satisfaction rating           0.000
Name: Talk Duration Seconds, dtype: float64


In [ ]:
# Business Insight

# Overall average talk duration is: 224.92 seconds = approximately 3 minutes 45 seconds
# The correlations are approximately: 
# Speed of Answer vs Talk Duration: -0.003 
# Satisfaction vs Talk Duration: 0.000

# There is meaningful variation in handling time by topic and agent. Admin Support, Contract-related and 
# Streaming calls have the longest average talk durations, while Payment-related calls are the shortest.

# This indicates that call type/complexity is a more useful area for investigating handling-time variation 
# than simply looking at how quickly calls are answered.

# At the agent level, Dan has the highest average talk duration and Diane the lowest, but this should not 
# automatically be interpreted as higher/lower performance, because agents may receive different mixes of topics.

In [ ]:
# Business Question 9: Can we predict customer satisfaction or operational outcomes?
# Analysis type: Predictive Analysis

In [ ]:
# We'll define:
#Low Satisfaction = rating ≤ 3

In [40]:
# pip install -U scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.2 MB 6.0 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/8.2 MB 4.7 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.2 MB 4.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.2 MB 4.3 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.2 MB 4.2 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.2 MB 4.1 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.2 MB 4.1 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.2 MB 4.1 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.2 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 4.0 MB/s  0:00:02
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
    --------------------------------------- 0.8/36.6 MB 3.9 MB/s eta 0:00:10
   - ----------------------


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

def convert_to_seconds(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, datetime.time):
        return (
            value.hour * 3600
            + value.minute * 60
            + value.second
            + value.microsecond / 1000000
        )

    return pd.to_timedelta(value).total_seconds()


df["Talk Duration Seconds"] = df["AvgTalkDuration"].apply(
    convert_to_seconds
)

# Only answered calls have satisfaction ratings
answered_df = df[df["Answered (Y/N)"] == "Y"].copy()

# Prediction target
answered_df["Low Satisfaction"] = (
    answered_df["Satisfaction rating"] <= 3
).astype(int)

# Features
X = answered_df[
    [
        "Speed of answer in seconds",
        "Talk Duration Seconds",
        "Resolved",
        "Topic",
        "Agent"
    ]
]

y = answered_df["Low Satisfaction"]

numeric_features = [
    "Speed of answer in seconds",
    "Talk Duration Seconds"
]

categorical_features = [
    "Resolved",
    "Topic",
    "Agent"
]

# Preprocessing
preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# Model
model = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter=1000)
        )
    ]
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train
model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

# Evaluation
accuracy = accuracy_score(y_test, predictions)

auc = roc_auc_score(y_test, probabilities)

print("Accuracy:", round(accuracy, 3))
print("ROC-AUC:", round(auc, 3))

Accuracy: 0.473
ROC-AUC: 0.46


In [ ]:
# Resolution: The model produces approximately:
# Accuracy: 47.3%
# ROC-AUC: 0.46

# A useful predictive model should perform meaningfully better than random classification. 
# Here, the model does not provide useful predictive power.

#Business Insight:

# The available operational data does not contain enough information to reliably predict low customer 
# satisfaction. That aligns with Question 6, where:
# Speed of Answer had almost zero correlation with satisfaction.
# Talk Duration had almost zero correlation with satisfaction.
# Topic and agent differences were relatively small.
# Therefore, we should not claim predictive capability from this dataset.

In [ ]:
# Business Question 10: What actions should management take based on the findings?
# Analysis type: Actionable Narrative

# 1. Investigate the decline in customer satisfaction
    # CSAT declined from 3.45 in January to 3.37 in March.
    # Speed of Answer remained broadly stable.
    # Therefore, investigate interaction quality, issue complexity, repeat contacts and customer expectations.

# 2. Investigate Streaming calls
    # Streaming had the lowest answered-call resolution rate at 88.43%.
    # Review the underlying reasons for unresolved Streaming calls.

# 3. Review high-duration call types
    # Admin Support, Contract-related and Streaming calls had the highest average talk duration.
    # Investigate whether these topics require additional knowledge, process changes or escalation support.

# 4. Use agent analysis for coaching, not ranking alone
    # Agent differences exist, but call mix may explain some variation.
    # Compare agents within the same topic/call type before making performance decisions.

# 5. Improve the data available for predictive analytics
    # The current dataset was insufficient to reliably predict customer satisfaction.
    # Add variables such as transfers, holds, repeat contacts, escalation, complaint type and customer history.

In [ ]:
# Contact-center performance weakened after January, with customer satisfaction declining continuously 
# despite relatively stable Speed of Answer. Topic-level analysis indicates greater variation in resolution 
# and handling time than agent-level analysis, particularly for Streaming, Admin Support and Contract-related 
# calls. The data does not support a meaningful relationship between response time, talk duration and 
# satisfaction, and predictive modeling did not achieve sufficient accuracy. Management should therefore 
# focus on topic-specific root-cause investigation, call complexity, interaction quality and richer operational 
# data rather than relying solely on response-time improvement.

In [ ]:
### To pull output data into excel

In [1]:
# ============================================================
# CONTACT CENTER OPERATIONAL ANALYTICS
# Q1-Q10 OUTPUT TABLES -> ONE EXCEL WORKBOOK
# ============================================================

import pandas as pd
import numpy as np
import datetime

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

input_file = "01 Call-Center-Dataset.xlsx"
#df = pd.read_excel(r"D:\Reading Material\Statistics\Descriptive Stats\Contact Center Operational Analytics\01 Call-Center-Dataset.xlsx", sheet_name = "Sheet1")


df = pd.read_excel(input_file)

# Convert Date column to datetime
df["Date"] = pd.to_datetime(df["Date"])

# ------------------------------------------------------------
# 2. CONVERT TALK DURATION TO SECONDS
# ------------------------------------------------------------

def convert_to_seconds(value):

    if pd.isna(value):
        return np.nan

    if isinstance(value, datetime.time):
        return (
            value.hour * 3600
            + value.minute * 60
            + value.second
            + value.microsecond / 1000000
        )

    return pd.to_timedelta(value).total_seconds()


df["Talk Duration Seconds"] = (
    df["AvgTalkDuration"]
    .apply(convert_to_seconds)
)

# ------------------------------------------------------------
# 3. ANSWERED CALLS DATASET
# ------------------------------------------------------------

answered_df = df[
    df["Answered (Y/N)"] == "Y"
].copy()

answered_df["Resolved Numeric"] = (
    answered_df["Resolved"] == "Y"
).astype(int)


# ============================================================
# Q1 — OVERALL CONTACT CENTER PERFORMANCE
# ============================================================

total_calls = len(df)

answered_calls = (
    df["Answered (Y/N)"] == "Y"
).sum()

unanswered_calls = (
    df["Answered (Y/N)"] == "N"
).sum()

resolved_calls = (
    df["Resolved"] == "Y"
).sum()

q1 = pd.DataFrame({

    "Metric": [
        "Total Calls",
        "Answered Calls",
        "Unanswered Calls",
        "Answer Rate",
        "Resolved Calls",
        "Overall Resolution Rate",
        "Resolution Rate - Answered Calls",
        "Average Speed of Answer (sec)",
        "Average Talk Duration (sec)",
        "Average Talk Duration (min)",
        "Average Satisfaction Rating"
    ],

    "Value": [

        total_calls,

        answered_calls,

        unanswered_calls,

        answered_calls / total_calls * 100,

        resolved_calls,

        resolved_calls / total_calls * 100,

        answered_df["Resolved Numeric"].mean() * 100,

        answered_df[
            "Speed of answer in seconds"
        ].mean(),

        answered_df[
            "Talk Duration Seconds"
        ].mean(),

        answered_df[
            "Talk Duration Seconds"
        ].mean() / 60,

        answered_df[
            "Satisfaction rating"
        ].mean()
    ]
})


q1 = q1.round(2)


# ============================================================
# Q2 — PERFORMANCE VARIABILITY
# ============================================================

q2 = pd.DataFrame({

    "Metric": [
        "Speed of Answer",
        "Talk Duration",
        "Satisfaction Rating"
    ],

    "Mean": [

        answered_df[
            "Speed of answer in seconds"
        ].mean(),

        answered_df[
            "Talk Duration Seconds"
        ].mean(),

        answered_df[
            "Satisfaction rating"
        ].mean()
    ],

    "Std Dev": [

        answered_df[
            "Speed of answer in seconds"
        ].std(),

        answered_df[
            "Talk Duration Seconds"
        ].std(),

        answered_df[
            "Satisfaction rating"
        ].std()
    ],

    "Variance": [

        answered_df[
            "Speed of answer in seconds"
        ].var(),

        answered_df[
            "Talk Duration Seconds"
        ].var(),

        answered_df[
            "Satisfaction rating"
        ].var()
    ],

    "Minimum": [

        answered_df[
            "Speed of answer in seconds"
        ].min(),

        answered_df[
            "Talk Duration Seconds"
        ].min(),

        answered_df[
            "Satisfaction rating"
        ].min()
    ],

    "Median": [

        answered_df[
            "Speed of answer in seconds"
        ].median(),

        answered_df[
            "Talk Duration Seconds"
        ].median(),

        answered_df[
            "Satisfaction rating"
        ].median()
    ],

    "Maximum": [

        answered_df[
            "Speed of answer in seconds"
        ].max(),

        answered_df[
            "Talk Duration Seconds"
        ].max(),

        answered_df[
            "Satisfaction rating"
        ].max()
    ]
})


q2 = q2.round(2)


# Operational percentiles
q2_percentiles = pd.DataFrame({

    "Metric": [
        "Speed of Answer",
        "Talk Duration"
    ],

    "P90": [

        answered_df[
            "Speed of answer in seconds"
        ].quantile(0.90),

        answered_df[
            "Talk Duration Seconds"
        ].quantile(0.90)
    ],

    "P95": [

        answered_df[
            "Speed of answer in seconds"
        ].quantile(0.95),

        answered_df[
            "Talk Duration Seconds"
        ].quantile(0.95)
    ]
})


q2_percentiles = q2_percentiles.round(2)


# ============================================================
# Q3 — MONTHLY PERFORMANCE TREND
# ============================================================

df["Month"] = df["Date"].dt.strftime("%b")

month_order = [
    "Jan",
    "Feb",
    "Mar"
]


# Monthly volume
monthly_volume = (
    df.groupby("Month")
    .agg(
        Total_Calls=("Call Id", "count"),
        Answered_Calls=("Answered (Y/N)", "count")
    )
    .reindex(month_order)
    .reset_index()
)

# Correct Answered Calls count
monthly_answered_count = (
    df[df["Answered (Y/N)"] == "Y"]
    .groupby("Month")
    .size()
    .reindex(month_order)
    .fillna(0)
)

monthly_resolved_count = (
    df[df["Resolved"] == "Y"]
    .groupby("Month")
    .size()
    .reindex(month_order)
    .fillna(0)
)

monthly_volume["Answered_Calls"] = (
    monthly_answered_count.values
)

monthly_volume["Resolved_Calls"] = (
    monthly_resolved_count.values
)


# Answered-call operational metrics
monthly_answered = (
    answered_df.assign(
        Month=answered_df["Date"].dt.strftime("%b")
    )
    .groupby("Month")
    .agg(
        Avg_Speed_of_Answer=(
            "Speed of answer in seconds",
            "mean"
        ),

        Avg_Talk_Duration=(
            "Talk Duration Seconds",
            "mean"
        ),

        Avg_CSAT=(
            "Satisfaction rating",
            "mean"
        )
    )
    .reindex(month_order)
    .reset_index()
)


q3 = monthly_volume.merge(
    monthly_answered,
    on="Month"
)


q3["Answer_Rate_%"] = (
    q3["Answered_Calls"]
    / q3["Total_Calls"]
    * 100
)


q3["Resolution_Rate_%"] = (
    q3["Resolved_Calls"]
    / q3["Total_Calls"]
    * 100
)


q3 = q3.round(2)


# ============================================================
# Q4 — AGENT PERFORMANCE
# ============================================================

q4 = (
    answered_df
    .groupby("Agent")
    .agg(

        Answered_Calls=(
            "Call Id",
            "count"
        ),

        Resolved_Calls=(
            "Resolved Numeric",
            "sum"
        ),

        Resolution_Rate=(
            "Resolved Numeric",
            "mean"
        ),

        Avg_Speed_of_Answer=(
            "Speed of answer in seconds",
            "mean"
        ),

        Avg_Talk_Duration=(
            "Talk Duration Seconds",
            "mean"
        ),

        Avg_CSAT=(
            "Satisfaction rating",
            "mean"
        )
    )
    .reset_index()
)


q4["Resolution_Rate_%"] = (
    q4["Resolution_Rate"] * 100
)


q4 = q4.drop(
    columns=["Resolution_Rate"]
)


q4 = q4.sort_values(
    "Resolution_Rate_%"
)


q4 = q4.round(2)


# Overall benchmark
q4_benchmark = pd.DataFrame({

    "Metric": [
        "Resolution Rate",
        "Average Speed of Answer",
        "Average Talk Duration",
        "Average CSAT"
    ],

    "Overall Benchmark": [

        answered_df[
            "Resolved Numeric"
        ].mean() * 100,

        answered_df[
            "Speed of answer in seconds"
        ].mean(),

        answered_df[
            "Talk Duration Seconds"
        ].mean(),

        answered_df[
            "Satisfaction rating"
        ].mean()
    ]
})


q4_benchmark = q4_benchmark.round(2)


# ============================================================
# Q5 — TOPIC PERFORMANCE
# ============================================================

q5 = (
    answered_df
    .groupby("Topic")
    .agg(

        Answered_Calls=(
            "Call Id",
            "count"
        ),

        Resolved_Calls=(
            "Resolved Numeric",
            "sum"
        ),

        Resolution_Rate=(
            "Resolved Numeric",
            "mean"
        ),

        Avg_Speed_of_Answer=(
            "Speed of answer in seconds",
            "mean"
        ),

        Avg_Talk_Duration=(
            "Talk Duration Seconds",
            "mean"
        ),

        Avg_CSAT=(
            "Satisfaction rating",
            "mean"
        )
    )
    .reset_index()
)


q5["Resolution_Rate_%"] = (
    q5["Resolution_Rate"] * 100
)


q5 = q5.drop(
    columns=["Resolution_Rate"]
)


q5 = q5.sort_values(
    "Resolution_Rate_%"
)


q5 = q5.round(2)


# ============================================================
# Q6 — CUSTOMER SATISFACTION DRIVERS
# ============================================================

q6_correlation = (
    answered_df[
        [
            "Speed of answer in seconds",
            "Talk Duration Seconds",
            "Satisfaction rating"
        ]
    ]
    .corr()["Satisfaction rating"]
    .drop("Satisfaction rating")
    .reset_index()
)


q6_correlation.columns = [
    "Variable",
    "Correlation_with_CSAT"
]


q6_correlation = q6_correlation.round(3)


# CSAT by Topic
q6_topic = (
    answered_df
    .groupby("Topic")[
        "Satisfaction rating"
    ]
    .agg(["count", "mean"])
    .reset_index()
)


q6_topic.columns = [
    "Topic",
    "Answered_Calls",
    "Average_CSAT"
]


q6_topic = q6_topic.sort_values(
    "Average_CSAT",
    ascending=False
)


q6_topic = q6_topic.round(2)


# CSAT by Agent
q6_agent = (
    answered_df
    .groupby("Agent")[
        "Satisfaction rating"
    ]
    .agg(["count", "mean"])
    .reset_index()
)


q6_agent.columns = [
    "Agent",
    "Answered_Calls",
    "Average_CSAT"
]


q6_agent = q6_agent.sort_values(
    "Average_CSAT",
    ascending=False
)


q6_agent = q6_agent.round(2)


# CSAT by Resolution
q6_resolution = (
    answered_df
    .groupby("Resolved")[
        "Satisfaction rating"
    ]
    .agg(["count", "mean"])
    .reset_index()
)


q6_resolution.columns = [
    "Resolved",
    "Answered_Calls",
    "Average_CSAT"
]


q6_resolution = q6_resolution.round(2)


# ============================================================
# Q7 — RESOLUTION DRIVERS
# ============================================================

q7_correlation = (
    answered_df[
        [
            "Speed of answer in seconds",
            "Talk Duration Seconds",
            "Satisfaction rating",
            "Resolved Numeric"
        ]
    ]
    .corr()["Resolved Numeric"]
    .drop("Resolved Numeric")
    .reset_index()
)


q7_correlation.columns = [
    "Variable",
    "Correlation_with_Resolution"
]


q7_correlation = q7_correlation.round(3)


# Resolution by Topic
q7_topic = (
    answered_df
    .groupby("Topic")[
        "Resolved Numeric"
    ]
    .mean()
    .mul(100)
    .reset_index(
        name="Resolution_Rate_%"
    )
)


q7_topic = q7_topic.sort_values(
    "Resolution_Rate_%"
)


q7_topic = q7_topic.round(2)


# Resolution by Agent
q7_agent = (
    answered_df
    .groupby("Agent")[
        "Resolved Numeric"
    ]
    .mean()
    .mul(100)
    .reset_index(
        name="Resolution_Rate_%"
    )
)


q7_agent = q7_agent.sort_values(
    "Resolution_Rate_%"
)


q7_agent = q7_agent.round(2)


# ============================================================
# Q8 — TALK DURATION / HANDLING TIME RCA
# ============================================================

# By Topic
q8_topic = (
    answered_df
    .groupby("Topic")[
        "Talk Duration Seconds"
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)


q8_topic.columns = [
    "Topic",
    "Answered_Calls",
    "Avg_Talk_Duration",
    "Median_Talk_Duration",
    "Std_Dev",
    "Minimum",
    "Maximum"
]


q8_topic = q8_topic.sort_values(
    "Avg_Talk_Duration",
    ascending=False
)


q8_topic = q8_topic.round(2)


# By Agent
q8_agent = (
    answered_df
    .groupby("Agent")[
        "Talk Duration Seconds"
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std"
    ])
    .reset_index()
)


q8_agent.columns = [
    "Agent",
    "Answered_Calls",
    "Avg_Talk_Duration",
    "Median_Talk_Duration",
    "Std_Dev"
]


q8_agent = q8_agent.sort_values(
    "Avg_Talk_Duration",
    ascending=False
)


q8_agent = q8_agent.round(2)


# Talk duration relationships
q8_correlation = (
    answered_df[
        [
            "Speed of answer in seconds",
            "Talk Duration Seconds",
            "Satisfaction rating"
        ]
    ]
    .corr()["Talk Duration Seconds"]
    .drop("Talk Duration Seconds")
    .reset_index()
)


q8_correlation.columns = [
    "Variable",
    "Correlation_with_Talk_Duration"
]


q8_correlation = q8_correlation.round(3)


# ============================================================
# Q9 — PREDICTIVE ANALYSIS
# ============================================================

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)


model_df = answered_df[
    [
        "Speed of answer in seconds",
        "Talk Duration Seconds",
        "Resolved",
        "Topic",
        "Agent",
        "Satisfaction rating"
    ]
].copy()


# Target
model_df["Low_Satisfaction"] = (
    model_df["Satisfaction rating"] <= 3
).astype(int)


X = model_df[
    [
        "Speed of answer in seconds",
        "Talk Duration Seconds",
        "Resolved",
        "Topic",
        "Agent"
    ]
]


y = model_df[
    "Low_Satisfaction"
]


numeric_features = [
    "Speed of answer in seconds",
    "Talk Duration Seconds"
]


categorical_features = [
    "Resolved",
    "Topic",
    "Agent"
]


numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "scaler",
        StandardScaler()
    )
])


categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])


preprocessor = ColumnTransformer([

    (
        "numeric",
        numeric_transformer,
        numeric_features
    ),

    (
        "categorical",
        categorical_transformer,
        categorical_features
    )
])


model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])


X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)


model.fit(
    X_train,
    y_train
)


predictions = model.predict(
    X_test
)


probabilities = model.predict_proba(
    X_test
)[:, 1]


accuracy = accuracy_score(
    y_test,
    predictions
)


roc_auc = roc_auc_score(
    y_test,
    probabilities
)


q9_model = pd.DataFrame({

    "Metric": [
        "Target",
        "Training Records",
        "Test Records",
        "Accuracy",
        "ROC-AUC"
    ],

    "Value": [

        "Low Satisfaction (Rating <= 3)",

        len(X_train),

        len(X_test),

        round(accuracy, 3),

        round(roc_auc, 3)
    ]
})


# Additional data required
q9_additional_data = pd.DataFrame({

    "Additional Data Recommended": [

        "Call reason / sub-reason",

        "Customer segment",

        "Previous contact history",

        "First-contact resolution",

        "Transfer count",

        "Hold duration",

        "Repeat-call indicator",

        "Escalation indicator",

        "Complaint indicator",

        "Customer tenure / value",

        "Interaction quality score"
    ]
})


# ============================================================
# Q10 — MANAGEMENT ACTION PLAN
# ============================================================

streaming_resolution = q7_topic.loc[
    q7_topic["Topic"] == "Streaming",
    "Resolution_Rate_%"
].iloc[0]


speed_csat_corr = q6_correlation.loc[
    q6_correlation["Variable"]
    == "Speed of answer in seconds",
    "Correlation_with_CSAT"
].iloc[0]


q10 = pd.DataFrame({

    "Finding": [

        "Customer satisfaction declined from January to March",

        "Streaming has the lowest resolution rate among topics",

        "Admin Support, Contract and Streaming have relatively high talk duration",

        "Agent-level differences exist but are relatively narrow",

        "Speed of Answer has almost no observed relationship with CSAT",

        "Current variables do not provide useful predictive power for low satisfaction"
    ],

    "Evidence": [

        "CSAT declined from 3.45 in Jan to 3.37 in Mar",

        f"Streaming resolution rate = {streaming_resolution:.2f}%",

        "These topics show higher average handling time",

        "Agent resolution rates vary within a relatively narrow range",

        f"Correlation between Speed of Answer and CSAT = {speed_csat_corr:.3f}",

        f"Current model ROC-AUC = {roc_auc:.3f}"
    ],

    "Management_Action": [

        "Investigate interaction quality, repeat contacts, escalations and issue complexity",

        "Conduct topic-level RCA on unresolved Streaming calls",

        "Review process complexity, documentation and dependency points",

        "Compare agents within similar topic/call mixes before taking performance action",

        "Avoid relying on response-time improvement alone to address CSAT",

        "Expand the dataset with richer customer and interaction-level variables"
    ]
})


# ============================================================
# EXPORT ALL OUTPUTS TO ONE EXCEL WORKBOOK
# ============================================================

output_file = "Contact_Center_Analysis_Outputs.xlsx"


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    q1.to_excel(
        writer,
        sheet_name="Q1_Overall_KPIs",
        index=False
    )

    q2.to_excel(
        writer,
        sheet_name="Q2_Variability",
        index=False
    )

    q2_percentiles.to_excel(
        writer,
        sheet_name="Q2_Percentiles",
        index=False
    )

    q3.to_excel(
        writer,
        sheet_name="Q3_Monthly_Trend",
        index=False
    )

    q4.to_excel(
        writer,
        sheet_name="Q4_Agent_Analysis",
        index=False
    )

    q4_benchmark.to_excel(
        writer,
        sheet_name="Q4_Agent_Benchmark",
        index=False
    )

    q5.to_excel(
        writer,
        sheet_name="Q5_Topic_Analysis",
        index=False
    )

    q6_correlation.to_excel(
        writer,
        sheet_name="Q6_CSAT_Correlation",
        index=False
    )

    q6_topic.to_excel(
        writer,
        sheet_name="Q6_CSAT_Topic",
        index=False
    )

    q6_agent.to_excel(
        writer,
        sheet_name="Q6_CSAT_Agent",
        index=False
    )

    q6_resolution.to_excel(
        writer,
        sheet_name="Q6_CSAT_Resolution",
        index=False
    )

    q7_correlation.to_excel(
        writer,
        sheet_name="Q7_Resolution_Corr",
        index=False
    )

    q7_topic.to_excel(
        writer,
        sheet_name="Q7_Resolution_Topic",
        index=False
    )

    q7_agent.to_excel(
        writer,
        sheet_name="Q7_Resolution_Agent",
        index=False
    )

    q8_topic.to_excel(
        writer,
        sheet_name="Q8_Talk_Topic_RCA",
        index=False
    )

    q8_agent.to_excel(
        writer,
        sheet_name="Q8_Talk_Agent_RCA",
        index=False
    )

    q8_correlation.to_excel(
        writer,
        sheet_name="Q8_Talk_Correlation",
        index=False
    )

    q9_model.to_excel(
        writer,
        sheet_name="Q9_Predictive_Model",
        index=False
    )

    q9_additional_data.to_excel(
        writer,
        sheet_name="Q9_Additional_Data",
        index=False
    )

    q10.to_excel(
        writer,
        sheet_name="Q10_Action_Plan",
        index=False
    )


print("Analysis completed successfully.")

print("\nExcel workbook created:")
print(output_file)

print("\nSheets created:")
print("1. Q1_Overall_KPIs")
print("2. Q2_Variability")
print("3. Q2_Percentiles")
print("4. Q3_Monthly_Trend")
print("5. Q4_Agent_Analysis")
print("6. Q4_Agent_Benchmark")
print("7. Q5_Topic_Analysis")
print("8. Q6_CSAT_Correlation")
print("9. Q6_CSAT_Topic")
print("10. Q6_CSAT_Agent")
print("11. Q6_CSAT_Resolution")
print("12. Q7_Resolution_Corr")
print("13. Q7_Resolution_Topic")
print("14. Q7_Resolution_Agent")
print("15. Q8_Talk_Topic_RCA")
print("16. Q8_Talk_Agent_RCA")
print("17. Q8_Talk_Correlation")
print("18. Q9_Predictive_Model")
print("19. Q9_Additional_Data")
print("20. Q10_Action_Plan")

Analysis completed successfully.

Excel workbook created:
Contact_Center_Analysis_Outputs.xlsx

Sheets created:
1. Q1_Overall_KPIs
2. Q2_Variability
3. Q2_Percentiles
4. Q3_Monthly_Trend
5. Q4_Agent_Analysis
6. Q4_Agent_Benchmark
7. Q5_Topic_Analysis
8. Q6_CSAT_Correlation
9. Q6_CSAT_Topic
10. Q6_CSAT_Agent
11. Q6_CSAT_Resolution
12. Q7_Resolution_Corr
13. Q7_Resolution_Topic
14. Q7_Resolution_Agent
15. Q8_Talk_Topic_RCA
16. Q8_Talk_Agent_RCA
17. Q8_Talk_Correlation
18. Q9_Predictive_Model
19. Q9_Additional_Data
20. Q10_Action_Plan
